# Итоговое задание: Анализ временных рядов
## Прогнозирование спроса на прокат велосипедов (Bike Sharing)

**Датасет**: daily.csv — ежедневные данные о прокате велосипедов (2011–2012)
**Целевая переменная**: `cnt` — общее число аренд за день
**Горизонт прогнозирования**: h=14 дней
**Метрики**: RMSE, MAE, MAPE


In [ ]:
import warnings, json, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')

BASE = Path('/home/user')
OUT = BASE / 'output'
OUT.mkdir(exist_ok=True)

---
## Задача №1. Подготовка данных и EDA

### Описание набора данных

Датасет содержит ежедневные данные о прокате велосипедов в системе Bike Sharing за период с 1 января 2011 по 31 декабря 2012 года (731 наблюдение).

**Переменные:**
- `instant` — индекс записи
- `dteday` — дата
- `season`, `yr`, `mnth`, `holiday`, `weekday`, `workingday`, `weathersit` — календарные и погодные признаки
- `temp`, `atemp`, `hum`, `windspeed` — нормализованные气象特征 (0–1)
- `casual`, `registered` — число аренд незарегистрированными/зарегистрированными пользователями
- `cnt` — целевая переменная (общее число аренд)

### Постановка задачи

- **Тип задачи**: одномерное прогнозирование временного ряда (univariate forecasting)
- **Горизонт**: h = 14 дней
- **Режим**: офлайн-прогнозирование
- **Метрики**: RMSE, MAE, MAPE

In [ ]:
# Загрузка данных
df = pd.read_csv(BASE / 'uploads' / 'day.csv')
df['ds'] = pd.to_datetime(df['dteday'])
df = df.sort_values('ds').reset_index(drop=True)
df['y'] = df['cnt']

print(f"Наблюдений: {len(df)}")
print(f"Период: {df['ds'].min()} — {df['ds'].max()}")
print(f"y: min={df['y'].min()}, max={df['y'].max()}, mean={df['y'].mean():.1f}, std={df['y'].std():.1f}")
print(f"\nПропуски:\n{df.isnull().sum()[df.isnull().sum()>0]}")
print(f"\nТипы данных:\n{df.dtypes}")
df.head()

In [ ]:
# Визуализация EDA
fig, axes = plt.subplots(3, 2, figsize=(16, 14))
fig.suptitle('EDA — Bike Sharing Dataset', fontsize=16, fontweight='bold')

# 1. Временной ряд
axes[0,0].plot(df['ds'], df['y'], lw=0.8, alpha=0.8)
axes[0,0].set_title('Daily Total Rentals (cnt)')
axes[0,0].set_ylabel('Count')

# 2. Гистограмма
axes[0,1].hist(df['y'], bins=40, edgecolor='black', alpha=0.7)
axes[0,1].set_title('Distribution of Daily Rentals')

# 3. По месяцам
dc = df.copy()
dc['month'] = dc['ds'].dt.month
sns.boxplot(data=dc, x='month', y='y', ax=axes[1,0])
axes[1,0].set_title('Distribution by Month')

# 4. По дням недели
sns.boxplot(data=dc, x='weekday', y='y', ax=axes[1,1])
axes[1,1].set_title('Distribution by Weekday')

# 5. Скользящее среднее
w=7; rm=df['y'].rolling(w).mean(); rs=df['y'].rolling(w).std()
axes[2,0].plot(df['ds'], df['y'], alpha=0.4, lw=0.5)
axes[2,0].plot(df['ds'], rm, c='red', lw=1.5)
axes[2,0].fill_between(df['ds'], rm-rs, rm+rs, color='red', alpha=0.2)
axes[2,0].set_title(f'Rolling Mean ({w}-day)')

# 6. Корреляции
sns.heatmap(df[['temp','atemp','hum','windspeed','casual','registered','y']].corr(),
            annot=True, fmt='.2f', cmap='RdBu_r', center=0, ax=axes[2,1], square=True)
axes[2,1].set_title('Correlation Matrix')

plt.tight_layout()
fig.savefig(OUT/'01_eda.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ADF-тест на стационарность
from statsmodels.tsa.stattools import adfuller

r = adfuller(df['y'].dropna())
print(f"ADF (original): stat={r[0]:.4f}, p={r[1]:.4f} → {'Нестационарный' if r[1]>0.05 else 'Стационарный'}")

d = df['y'].diff().dropna()
rd = adfuller(d)
print(f"ADF (1st diff): stat={rd[0]:.4f}, p={rd[1]:.2e} → {'Стационарный' if rd[1]<0.05 else 'Нестационарный'}")

# Декомпозиция
from statsmodels.tsa.seasonal import seasonal_decompose
decomp = seasonal_decompose(df['y'], model='additive', period=7)
fig2, ax2 = plt.subplots(4,1,figsize=(14,10))
decomp.observed.plot(ax=ax2[0], title='Observed')
decomp.trend.plot(ax=ax2[1], title='Trend')
decomp.seasonal.plot(ax=ax2[2], title='Seasonal (period=7)')
decomp.resid.plot(ax=ax2[3], title='Residual')
plt.tight_layout()
fig2.savefig(OUT/'01_decomp.png', dpi=150, bbox_inches='tight')
plt.show()

### Выводы по EDA (Задача №1)

1. **Нестационарность**: Ряд нестационарен (ADF p=0.34), но первая разность стационарна (p<0.001). Это означает необходимость дифференцирования или моделей, работающих с трендом.
2. **Сезонность**: Явная недельная сезонность, подтверждённая декомпозицией.
3. **Корреляции**: Сильная связь с `registered` (r=0.95) и `temp` (r=0.63).
4. **Тренд**: Наблюдается восходящий тренд с сезонными пиками летом.
5. **Подготовленные данные**: 731 наблюдение, без пропусков, целевая переменная `y=cnt`.

---
## Задача №2. Статистические методы (statsforecast)

### Используемые модели (10 методов)

| Модель | Тип | Параметры |
|--------|-----|----------|
| AutoARIMA | Статистическая (авто) | Автоматический подбор (p,d,q)(P,D,Q) |
| ARIMA(1,1,1) | Статистическая (ручная) | order=(1,1,1), season_length=7 |
| AutoETS | Статистическая (авто) | Автоматический подбор error/trend/season |
| Holt | Статистическая (ручная) | season_length=7 |
| AutoTheta | Статистическая (авто) | Автоматический параметр theta |
| Theta | Статистическая (ручная) | season_length=7 |
| HoltWinters | Статистическая (ручная) | season_length=7 |
| HistoricAverage | Бейзлайн | Среднее по истории |
| SeasonalNaive | Бейзлайн | Последнее значение того же дня недели |
| Naive | Бейзлайн | Последнее наблюдение |

In [ ]:
# Метрики
def calc_metrics(actual, predicted):
    a, p = np.array(actual, dtype=float), np.array(predicted, dtype=float)
    rmse = float(np.sqrt(np.mean((a-p)**2)))
    mae = float(np.mean(np.abs(a-p)))
    mask = a != 0
    mape = float(np.mean(np.abs((a[mask]-p[mask])/a[mask]))) * 100
    return rmse, mae, mape

h = 14
from statsforecast import StatsForecast
from statsforecast.models import (AutoARIMA, ARIMA, AutoETS, AutoTheta,
    Theta, HoltWinters, HistoricAverage, Naive, SeasonalNaive, Holt)

sdf = df[['ds','y']].copy()
sdf['unique_id'] = 'bike'
train = sdf.iloc[:-h]
test = sdf.iloc[-h:]

models = [
    AutoARIMA(season_length=7), ARIMA(order=(1,1,1), season_length=7),
    AutoETS(season_length=7), Holt(season_length=7),
    AutoTheta(season_length=7), Theta(season_length=7),
    HoltWinters(season_length=7), HistoricAverage(),
    SeasonalNaive(season_length=7), Naive(),
]
col_names = ['AutoARIMA','ARIMA','AutoETS','Holt','AutoTheta','Theta',
             'HoltWinters','HistoricAverage','SeasonalNaive','Naive']

sf = StatsForecast(models=models, freq='D', n_jobs=-1)
sf.fit(train)
fc = sf.predict(h=h)

In [ ]:
# Метрики на тесте
act = test['y'].values
fcd = fc.copy()
fcd['ds'] = pd.date_range(test['ds'].iloc[0], periods=h, freq='D')
fcd = fcd.merge(test[['ds','y']].reset_index(drop=True), on='ds')

print(f"{'Модель':<20} {'RMSE':>10} {'MAE':>10} {'MAPE%':>10}")
print("-"*52)
tm = {}
for col in col_names:
    if col in fc.columns:
        r, m, mp = calc_metrics(act, fcd[col].values)
        tm[col] = {'rmse':r, 'mae':m, 'mape':mp}
        print(f"{col:<20} {r:>10.2f} {m:>10.2f} {mp:>10.2f}")

best_test = min(tm, key=lambda x: tm[x]['rmse'])
print(f"\nЛучшая на тесте: {best_test} (RMSE={tm[best_test]['rmse']:.2f})")

In [ ]:
# Бектестинг (Cross-Validation)
cv = sf.cross_validation(df=train, h=14, step_size=30, n_windows=3)
cutoffs = cv['cutoff'].unique()

print(f"{'Модель':<20} {'RMSE':>10} {'MAE':>10} {'MAPE%':>10}")
print("-"*52)
cvm = {}
for col in col_names:
    if col in cv.columns:
        ar, am, amp = [], [], []
        for c in cutoffs:
            mask = cv['cutoff'] == c
            r, m, mp = calc_metrics(cv.loc[mask,'y'].values, cv.loc[mask,col].values)
            ar.append(r); am.append(m); amp.append(mp)
        cvm[col] = {'rmse':np.mean(ar), 'mae':np.mean(am), 'mape':np.mean(amp)}
        print(f"{col:<20} {cvm[col]['rmse']:>10.2f} {cvm[col]['mae']:>10.2f} {cvm[col]['mape']:>10.2f}")

best_cv = min(cvm, key=lambda x: cvm[x]['rmse'])
print(f"\nЛучшая по CV: {best_cv} (RMSE={cvm[best_cv]['rmse']:.2f})")

In [ ]:
# Анализ остатков лучшей модели
resids = act - fcd[best_test].values
fig, ax = plt.subplots(1,3,figsize=(15,4))
ax[0].hist(resids, bins=20, edgecolor='k', alpha=0.7)
ax[0].axvline(0, c='r', ls='--')
ax[0].set_title(f'{best_test}: Residuals')
ax[1].plot(resids, 'o-', ms=4); ax[1].axhline(0, c='r', ls='--')
ax[1].set_title('Residuals over Time')
from statsmodels.graphics.tsaplots import plot_acf
plot_acf(resids, ax=ax[2], lags=10); ax[2].set_title('ACF')
plt.tight_layout()
fig.savefig(OUT/'02_resid.png', dpi=150, bbox_inches='tight')
plt.show()

### Выводы по статистическим моделям (Задача №2)

1. **Лучшая на тесте**: HistoricAverage (RMSE=2527.79) — простое усреднение оказалось эффективным из-за относительно стабильного тренда.
2. **Лучшая по CV**: ARIMA(1,1,1) (RMSE=882.39) — модель с дифференцированием лучше обобщает.
3. **Бейзлайны конкурентоспособны**: Naive и HistoricAverage показывают результаты лучше сложных моделей на тесте, что говорит о слабой предсказуемой структуре ряда.
4. **Остатки**: Не полностью нормальны, ACF показывает остаточную автокорреляцию — есть потенциал для улучшения.

---
## Задача №3. ML и DL методы

### Feature Engineering (mlforecast)

Для ML-моделей созданы следующие признаки:
- **Календарные**: day_of_week, day_of_month, month, year, day_of_year, is_weekend
- **Лаговые**: lag_1, lag_2, lag_3, lag_7, lag_14, lag_21, lag_28, lag_30
- **Скользящие статистики**: rolling_mean(7,14,28), rolling_std(7,14,28)

### Модели

| Категория | Модель | Обоснование |
|-----------|--------|-------------|
| ML | RandomForest | Устойчив к переобучению, не требует масштабирования |
| ML | XGBoost | Градиентный бустинг — лидер на табличных данных |
| ML | LightGBM | Быстрый градиентный бустинг с гистограммами |
| DL | N-BEATSx | State-of-the-art архитектура для ВР |
| DL | N-HiTS | Иерархическая интерполяция для ВР |
| DL | LSTM | Рекуррентная сеть для последовательностей |

In [ ]:
# Feature Engineering
df_f = df[['ds','y']].copy()
df_f['dow']   = df_f['ds'].dt.dayofweek
df_f['dom']   = df_f['ds'].dt.day
df_f['month'] = df_f['ds'].dt.month
df_f['year']  = df_f['ds'].dt.year
df_f['doy']   = df_f['ds'].dt.dayofyear
df_f['wknd']  = (df_f['dow'] >= 5).astype(int)
for lag in [1,2,3,7,14,21,28,30]:
    df_f[f'lag_{lag}'] = df_f['y'].shift(lag)
for w in [7,14,28]:
    df_f[f'rm{w}'] = df_f['y'].rolling(w).mean()
    df_f[f'rs{w}'] = df_f['y'].rolling(w).std()
df_f = df_f.dropna().reset_index(drop=True)

tr = df_f.iloc[:-h]
te = df_f.iloc[-h:]
feats = [c for c in tr.columns if c not in ['ds','y']]
print(f"Признаки: {len(feats)}, Train: {len(tr)}, Test: {len(te)}")

In [ ]:
# ML модели
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb
import lightgbm as lgb

rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(tr[feats], tr['y'])

xgb_m = xgb.XGBRegressor(n_estimators=200, max_depth=5, learning_rate=0.1,
                          random_state=42, n_jobs=-1)
xgb_m.fit(tr[feats], tr['y'])

lgb_m = lgb.LGBMRegressor(n_estimators=200, max_depth=5, learning_rate=0.1,
                           random_state=42, n_jobs=-1, verbose=-1)
lgb_m.fit(tr[feats], tr['y'])

# Рекурсивный прогноз
def rec_pred(mdl, tr, te, feats, h):
    hist = list(tr['y'].values)
    preds = []
    for i in range(h):
        rd = {}
        for f in feats:
            if f in ['dow','dom','month','year','doy','wknd']:
                rd[f] = te.iloc[i][f]
            elif f.startswith('lag_'):
                l = int(f.split('_')[1])
                rd[f] = hist[-l] if l <= len(hist) else hist[0]
            elif f.startswith('rm'):
                w = int(f[2:])
                rd[f] = np.mean(hist[-w:]) if len(hist) >= w else np.mean(hist)
            elif f.startswith('rs'):
                w = int(f[2:])
                rd[f] = np.std(hist[-w:]) if len(hist) >= w else 0
        p = mdl.predict(pd.DataFrame([rd])[feats])[0]
        preds.append(p)
        hist.append(p)
    return np.array(preds)

mlr = {}
act = te['y'].values
for nm, mdl in [('RandomForest', rf), ('XGBoost', xgb_m), ('LightGBM', lgb_m)]:
    p = rec_pred(mdl, tr, te, feats, h)
    r, m, mp = calc_metrics(act, p)
    mlr[nm] = {'rmse':r, 'mae':m, 'mape':mp, 'preds':p.tolist()}
    print(f"{nm}: RMSE={r:.2f}, MAE={m:.2f}, MAPE={mp:.2f}%")

In [ ]:
# DL модели (NeuralForecast)
import torch
torch.set_num_threads(2)

from neuralforecast import NeuralForecast
from neuralforecast.models import NHITS, NBEATSx, LSTM

nf_df = df[['ds','y']].copy()
nf_df['unique_id'] = 'bike'
nf_tr = nf_df.iloc[:-h]
nf_te = nf_df.iloc[-h:]

dl_models = [
    NHITS(h=h, input_size=28, max_steps=50, learning_rate=1e-3,
          scaler_type='standard', random_seed=42, batch_size=64),
    NBEATSx(h=h, input_size=28, max_steps=50, learning_rate=1e-3,
            scaler_type='standard', random_seed=42, batch_size=64),
    LSTM(h=h, input_size=28, max_steps=50, learning_rate=1e-3,
         scaler_type='standard', random_seed=42, batch_size=64),
]
dl_names = ['NHITS', 'NBEATSx', 'LSTM']

nf = NeuralForecast(models=dl_models, freq='D')
nf.fit(nf_tr)
dl_fc = nf.predict().reset_index(drop=True)
dl_fc['ds'] = pd.date_range(nf_te['ds'].iloc[0], periods=h, freq='D')

dlr = {}
act_dl = nf_te['y'].values
for nm in dl_names:
    if nm in dl_fc.columns:
        p = dl_fc[nm].values[:h]
        r, m, mp = calc_metrics(act_dl, p)
        dlr[nm] = {'rmse':r, 'mae':m, 'mape':mp, 'preds':p.tolist()}
        print(f"{nm}: RMSE={r:.2f}, MAE={m:.2f}, MAPE={mp:.2f}%")

In [ ]:
# Сводная таблица
print(f"{'Модель':<20} {'RMSE':>10} {'MAE':>10} {'MAPE%':>10}")
print("-"*52)
all_ml_dl = {**mlr, **dlr}
for nm, res in sorted(all_ml_dl.items(), key=lambda x: x[1]['rmse']):
    print(f"{nm:<20} {res['rmse']:>10.2f} {res['mae']:>10.2f} {res['mape']:>10.2f}")

# График
fig, ax = plt.subplots(figsize=(14,6))
ax.plot(range(h), act_dl, 'ko-', label='Actual', lw=2, ms=6)
cm_ml = ['blue','green','orange']
cm_dl = ['purple','brown','cyan']
for i, (nm, res) in enumerate(mlr.items()):
    ax.plot(range(h), res['preds'], '--', c=cm_ml[i],
            label=f"{nm} ({res['rmse']:.0f})", alpha=0.8)
for i, (nm, res) in enumerate(dlr.items()):
    ax.plot(range(h), res['preds'], '-.', c=cm_dl[i],
            label=f"{nm} ({res['rmse']:.0f})", alpha=0.8)
ax.set_title('ML & DL Forecast Comparison (h=14)')
ax.legend(fontsize=8, ncol=3); ax.grid(True, alpha=0.3)
plt.tight_layout()
fig.savefig(OUT/'03_mldl.png', dpi=150, bbox_inches='tight')
plt.show()

### Выводы по ML/DL (Задача №3)

1. **Лучшая ML**: LightGBM (RMSE=2442.96) — градиентный бустинг с гистограммами показал наилучший результат.
2. **Лучшая DL**: NBEATSx (RMSE=2767.46) — архитектура N-BEATS адаптирована для временных рядов.
3. **ML > DL**: На данном наборе данных (731 наблюдение) ML-модели превзошли DL-модели, что типично для малых выборок.
4. **LSTM показал худший результат** — рекуррентные сети требуют больше данных и более тонкой настройки.

---
## Задача №4. Пайплайн и тестирование

### Описание пайплайна

Пайплайн автоматизирует:
1. **Выбор модели** через кросс-валидацию (3 окна, шаг=30)
2. **Обучение** финальных кандидатов
3. **Инференс** и оценку на тесте
4. **Визуализацию** результатов

In [ ]:
from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA, AutoETS, AutoTheta, SeasonalNaive

sdf2 = df[['ds','y']].copy()
sdf2['unique_id'] = 'bike'
train2 = sdf2.iloc[:-h]
test2 = sdf2.iloc[-h:]

# CV для выбора лучшей модели
candidates = [
    ('AutoARIMA', AutoARIMA(season_length=7)),
    ('AutoETS', AutoETS(season_length=7)),
    ('AutoTheta', AutoTheta(season_length=7)),
    ('SeasonalNaive', SeasonalNaive(season_length=7)),
]

bs = float('inf'); bn = None; cvs = {}
print("Model selection via CV...")
for nm, mdl in candidates:
    sf = StatsForecast(models=[mdl], freq='D', n_jobs=1)
    t0 = time.time()
    cv = sf.cross_validation(df=train2, h=14, step_size=30, n_windows=3)
    elapsed = time.time() - t0
    rs = []
    for c in cv['cutoff'].unique():
        mask = cv['cutoff'] == c
        r, _, _ = calc_metrics(cv.loc[mask,'y'].values, cv.loc[mask,nm].values)
        rs.append(r)
    avg = np.mean(rs)
    cvs[nm] = {'rmse': avg, 'time': elapsed}
    print(f"  {nm}: CV RMSE={avg:.2f}, time={elapsed:.1f}s")
    if avg < bs:
        bs = avg; bn = nm

print(f"\nSelected: {bn} (CV RMSE={bs:.2f})")

In [ ]:
# Финальное обучение и прогноз
sf_f = StatsForecast(
    models=[AutoARIMA(season_length=7), AutoETS(season_length=7),
            AutoTheta(season_length=7), SeasonalNaive(season_length=7)],
    freq='D', n_jobs=1
)
sf_f.fit(train2)
t0 = time.time()
fc_f = sf_f.predict(h=h)
inf_t = time.time() - t0

fcd2 = fc_f.copy()
fcd2['ds'] = pd.date_range(test2['ds'].iloc[0], periods=h, freq='D')
fcd2 = fcd2.merge(test2[['ds','y']].reset_index(drop=True), on='ds')
act2 = test2['y'].values

print(f"{'Модель':<20} {'RMSE':>10} {'MAE':>10} {'MAPE%':>10}")
print("-"*52)
ptm = {}
for col in ['AutoARIMA','AutoETS','AutoTheta','SeasonalNaive']:
    if col in fc_f.columns:
        r, m, mp = calc_metrics(act2, fcd2[col].values)
        ptm[col] = {'rmse':r, 'mae':m, 'mape':mp}
        print(f"{col:<20} {r:>10.2f} {m:>10.2f} {mp:>10.2f}")

# График
fig, ax = plt.subplots(figsize=(12,5))
ax.plot(train2['ds'], train2['y'], label='Train', c='k', lw=1)
ax.plot(test2['ds'], test2['y'], label='Test', c='r', lw=2, marker='o')
for col in ['AutoARIMA','AutoETS','AutoTheta','SeasonalNaive']:
    if col in fc_f.columns:
        ax.plot(fcd2['ds'], fcd2[col], label=col, lw=1.5, alpha=0.8)
ax.axvline(train2['ds'].iloc[-1], c='gray', ls='--', alpha=0.5)
ax.set_title(f'Pipeline Forecast (best={bn})')
ax.legend(fontsize=9)
plt.tight_layout()
fig.savefig(OUT/'04_pipeline.png', dpi=150, bbox_inches='tight')
plt.show()

### Выводы по пайплайну (Задача №4)

1. **Лучшая модель по CV**: AutoTheta (RMSE=888.17) — быстрая и точная.
2. **Производительность**: Время CV ~18s, инференс <0.01s.
3. **Автоматизация**: Пайплайн полностью автоматизирует выбор, обучение и оценку.
4. **Надёжность**: Кросс-валидация на 3 окнах обеспечивает устойчивую оценку.

---
## Общее заключение

**Датасет**: Bike Sharing (731 наблюдение, 2011-01-01 — 2012-12-31)

**Цель**: Прогнозирование ежедневного спроса на прокат велосипедов (h=14)

### Итоговая таблица результатов

| Модель | Тип | RMSE (тест) | RMSE (CV) |
|--------|-----|-------------|----------|
| Naive | Бейзлайн | 2561.43 | 1146.08 |
| HistoricAverage | Бейзлайн | 2527.79 | 1672.16 |
| SeasonalNaive | Бейзлайн | 2894.87 | 1759.03 |
| AutoARIMA | Стат. (авто) | 2779.37 | 928.81 |
| ARIMA(1,1,1) | Стат. (ручной) | 2882.77 | 882.39 |
| AutoETS | Стат. (авто) | 3002.30 | 925.78 |
| Holt | Стат. (ручной) | 2856.57 | 930.54 |
| AutoTheta | Стат. (авто) | 2845.17 | 888.17 |
| Theta | Стат. (ручной) | 2825.78 | 886.72 |
| HoltWinters | Стат. (ручной) | 2850.38 | 946.68 |
| **LightGBM** | **ML** | **2442.96** | — |
| XGBoost | ML | 2453.50 | — |
| RandomForest | ML | 2675.62 | — |
| NBEATSx | DL | 2767.46 | — |
| NHITS | DL | 2860.30 | — |
| LSTM | DL | 3152.89 | — |

### Ключевые выводы

1. **ML-модели (LightGBM) показали лучший результат на тесте** (RMSE=2442.96), превзойдя все статистические и DL-модели.
2. **По CV лучшая статистическая модель — ARIMA(1,1,1)** (RMSE=882.39), что говорит о лучшей обобщающей способности.
3. **DL-модели уступают ML** на малой выборке (731 наблюдение), что ожидаемо.
4. **Простые бейзлайны конкурентоспособны** — ряд имеет слабую предсказуемую структуру.
5. **Пайплайн** автоматизирует выбор модели через CV и обеспечивает воспроизводимость.

### Рекомендации

- Для production-системы рекомендуем **LightGBM** (лучший RMSE на тесте) или **AutoTheta** (лучший баланс точности/скорости по CV).
- Для повышения точности: добавить внешние признаки (погода, события), увеличить горизонт для DL-моделей.